# Data Acquisition Notebook
Load Electric Vehicle CSV, Gas GeJSON, tract layer, and demographics table

In [1]:
# Imports
import sys
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == 'notebooks' else cwd
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))


from src.ingest.fetch_afdc import (
    fetch_afdc_stations,
    save_raw as save_afdc_raw,
    quality_control as quality_control_afdc,
)
from src.ingest.fetch_gas_stations import (
    fetch_gas,
    save_raw as save_gas_raw,
    qc_summary as quality_control_gas,
)
from src.ingest.fetch_census import (
    fetch_acs_ca_demographics_2024
)
from src.config import RAW_DIR, EXTERNAL_DIR

print(f'Project root: {project_root}')
print(f'Raw data directory: {RAW_DIR}')

Project root: c:\Users\caden\Documents\CA EV Analytics
Raw data directory: C:\Users\caden\Documents\CA EV Analytics\data\raw


## Pull EV and Gas Data
Run only when refreshing raw data

In [2]:
# Electric
ev = fetch_afdc_stations()
save_afdc_raw(ev)
quality_control_afdc(ev)

# Gas
gas = fetch_gas()
save_gas_raw(gas)
quality_control_gas(gas)

# Census
census = fetch_acs_ca_demographics_2024()

Fetched 21311 stations from AFDC (station_counts total_results=78506)
Saved raw JSON : C:\Users\caden\Documents\CA EV Analytics\data\raw\afdc_ev_stations_2026-07-08.json
Saved flat csv: C:\Users\caden\Documents\CA EV Analytics\data\raw\afdc_ev_stations_2026-07-08.csv

--- QC Summary ---
 Total records:  21311
Valid lat/lon:   21311 (100.0%)
 Non-California rows:  0
Access Code Breakdown: 
 access_code
public     20081
private     1230
DFDC ports: 19160
  offset=     0  got 2000 features  (running total 2000)
  offset=  2000  got 2000 features  (running total 4000)
  offset=  4000  got 2000 features  (running total 6000)
  offset=  6000  got 2000 features  (running total 8000)
  offset=  8000  got 1617 features  (running total 9617)
Saved raw GeoJson: C:\Users\caden\Documents\CA EV Analytics\data\raw\cec_gas_stations_2026-07-08.geojson
Saved flat CSV: C:\Users\caden\Documents\CA EV Analytics\data\raw\cec_gas_stations_2026-07-08.csv

--- QC summary ---
Total records:     9617
Valid geome

## Locate latest raw files

In [3]:
from src.config import RAW_DIR

# CSV
ev_csv_files = sorted(RAW_DIR.glob("afdc_ev_stations_*.csv"))
gas_csv_files = sorted(RAW_DIR.glob("cec_gas_stations_*.csv"))
census_csv_files = sorted(RAW_DIR.glob("census_data_*.csv"))

latest_ev_csv = ev_csv_files[-1] if len(ev_csv_files) > 0 else None
latest_gas_csv = gas_csv_files[-1] if len(gas_csv_files) > 0 else None
latest_census_csv = census_csv_files[-1] if len(census_csv_files) > 0 else None

# JSON / GeoJSON
ev_json_files = sorted(RAW_DIR.glob('afdc_ev_stations_*.json'))
gas_json_files = sorted(RAW_DIR.glob('cec_gas_stations_*.geojson'))

latest_ev_json = ev_json_files[-1] if len(ev_json_files) > 0 else None
latest_gas_json = gas_json_files[-1] if len(gas_json_files) > 0 else None

# Validation Check
print(f"Latest EV CSV: {latest_ev_csv} \n Latest EV JSON: {latest_ev_json}")
print(f"Latest Gas CSV: {latest_gas_csv} \n Latest Gas JSON: {latest_gas_json}")
print(f"Latest Census CSV: {latest_census_csv}")

Latest EV CSV: C:\Users\caden\Documents\CA EV Analytics\data\raw\afdc_ev_stations_2026-07-08.csv 
 Latest EV JSON: C:\Users\caden\Documents\CA EV Analytics\data\raw\afdc_ev_stations_2026-07-08.json
Latest Gas CSV: C:\Users\caden\Documents\CA EV Analytics\data\raw\cec_gas_stations_2026-07-08.csv 
 Latest Gas JSON: C:\Users\caden\Documents\CA EV Analytics\data\raw\cec_gas_stations_2026-07-08.geojson
Latest Census CSV: C:\Users\caden\Documents\CA EV Analytics\data\raw\census_data_2026-07-08.csv


## Load raw files

In [4]:
# EV
ev_csv = pd.read_csv(latest_ev_csv)
ev_json = pd.read_json(latest_ev_json)

# Gas
gas_csv = pd.read_csv(latest_gas_csv)
gas_json = gpd.read_file(latest_gas_json)

# Census
census_csv = pd.read_csv(latest_census_csv)

## Detailed Raw Data Quality Checks

In [5]:
# Define California Bounding Box bounds
CA_MIN_LAT, CA_MAX_LAT = 32.5, 42.1
CA_MIN_LON, CA_MAX_LON = -124.5, -114.1

def run_interactive_qc(df, df_name, lat_col='latitude', lon_col='longitude', id_col='id'):
    print(f"=== QC Report for {df_name} ===")
    print(f"Total records: {len(df)}")
    
    # Duplicates
    if id_col in df.columns:
        dup_count = df.duplicated(subset=[id_col]).sum()
        print(f"Duplicate records (by {id_col}): {dup_count} ({dup_count/len(df):.1%})")
    
    # Missing Coordinates
    missing_coords = df[[lat_col, lon_col]].isnull().any(axis=1).sum()
    print(f"Missing lat/lon coordinates: {missing_coords} ({missing_coords/len(df):.1%})")
    
    # Coordinate range validity (California check)
    valid_coords = df[[lat_col, lon_col]].dropna()
    out_of_bounds = valid_coords[
        (valid_coords[lat_col] < CA_MIN_LAT) | (valid_coords[lat_col] > CA_MAX_LAT) |
        (valid_coords[lon_col] < CA_MIN_LON) | (valid_coords[lon_col] > CA_MAX_LON)
    ]
    print(f"Coordinates outside CA bounding box: {len(out_of_bounds)} ({len(out_of_bounds)/len(df):.1%})")
    
    # Missing counts overall
    print("\nMissing values count per column:")
    print(df.isnull().sum())
    print("\n" + "="*40 + "\n")

# Run QC on EV data
run_interactive_qc(ev_csv, "EV Stations (CSV)")

# Run QC on Gas stations data
run_interactive_qc(gas_csv, "Gas Stations (CSV)", id_col='OBJECTID')

# Run QC on Census data
run_interactive_qc(census_csv, "Census (CSV)", id_col='GEOID')


=== QC Report for EV Stations (CSV) ===
Total records: 21311
Duplicate records (by id): 0 (0.0%)
Missing lat/lon coordinates: 0 (0.0%)
Coordinates outside CA bounding box: 0 (0.0%)

Missing values count per column:
id                         0
station_name               0
street_address             5
city                       0
state                      0
zip                        0
latitude                   0
longitude                  0
status_code                0
access_code                0
ev_connector_types         4
ev_dc_fast_num         18442
ev_level1_evse_num     21195
ev_level2_evse_num      2626
ev_network                 0
ev_pricing             19462
facility_type          17037
funding_sources        21310
date_last_confirmed       81
open_date                103
owner_type_code        16763
dtype: int64


=== QC Report for Gas Stations (CSV) ===
Total records: 9617
Duplicate records (by OBJECTID): 0 (0.0%)
Missing lat/lon coordinates: 3 (0.0%)
Coordinates outside 

KeyError: "None of [Index(['latitude', 'longitude'], dtype='object')] are in the [columns]"